In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

In [18]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [19]:
categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["number_of_courses_viewed", "annual_income", "interaction_count", "lead_score"]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

In [20]:
df["industry"].mode()[0]

'technology'

In [21]:
corr = df[numerical].corr()
corr

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.161300,0.721609,0.757204
annual_income,0.161300,1.000000,0.122842,0.229496
interaction_count,0.721609,0.122842,1.000000,0.915746
lead_score,0.757204,0.229496,0.915746,1.000000


In [22]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for a, b in pairs:
    print(a, b, round(corr.loc[a, b], 4))

interaction_count lead_score 0.9157
number_of_courses_viewed lead_score 0.7572
number_of_courses_viewed interaction_count 0.7216
annual_income interaction_count 0.1228


In [23]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train["converted"].values
y_val = df_val["converted"].values

df_train = df_train.drop(columns="converted")
df_val = df_val.drop(columns="converted")

In [24]:
for col in categorical:
    print(col, round(mutual_info_score(y_train, df_train[col]), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [25]:
features = categorical + numerical

def fit_accuracy(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feats].to_dict(orient="records"))
    X_val = dv.transform(df_val[feats].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

In [26]:
base = fit_accuracy(features)
base, round(base, 2)

(0.645, 0.65)

In [27]:
for f in ["lead_source", "number_of_courses_viewed", "interaction_count"]:
    acc = fit_accuracy([x for x in features if x != f])
    print(f, acc, base - acc)

lead_source 0.642 0.0030000000000000027
number_of_courses_viewed 0.643 0.0020000000000000018
interaction_count 0.601 0.04400000000000004


In [28]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(fit_accuracy(features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
